# Microsolvation: A Few Real Water Molecules

:::{div}
:class: colab-badge
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blinkletter/UsingPsi4/blob/main/book/10-Solvent/104-Microsolvation.ipynb)
:::

In the [first notebook](101-Solvent.ipynb) of this chapter we found out the hard way that *Psi4* cannot follow a reaction path in a **continuum** solvent model, because it has no exact gradients for the solvent. So let us try the other way to put water into a calculation: add some **real water molecules**. This is called **microsolvation**, or an **explicit solvent** model.

The idea is simple. Chloride is a hydrogen-bond acceptor, and in water each chloride is surrounded by water molecules whose O&ndash;H bonds point at it. If we place a few water molecules around our *S<sub>N</sub>2* transition state, each one hydrogen-bonded to a chloride, the calculation is just an ordinary B3LYP calculation on a bigger molecule. *Psi4* has exact gradients for that, so the transition state search, the frequencies and the IRC all work exactly as they did in gas phase.

Of course four water molecules are not an ocean. They capture the **specific** interactions (the hydrogen bonds to the chloride) but not the **bulk** dielectric of the solvent beyond them, and the answer may depend on how many waters we use and where we put them. That is the price. The continuum model is the opposite: bulk dielectric, but no hydrogen bonds. Real solvents do both.

We will run the same calculations for the bare system in gas phase, so that we can compare the two reaction paths directly.

## Learning Goals

This exercise will hopefully accomplish the following goals while learning new skills:
-	We will **build** a symmetric cluster of [Cl&ndash;CH<sub>3</sub>&ndash;Cl]<sup>&minus;</sup> with four water molecules, with a few lines of *Python*.
-	We will let one **mirror plane** hold the cluster on the saddle point while the waters find their own best arrangement, check the **vibrations**, and then do a real **transition state search** without symmetry.
-	We will follow the **IRC** in both directions, clean up its ends, and **optimize** the complexes at the bottom of the valley.
-	We will do the same for the bare system in gas phase and **compare** the energy profiles and the bond lengths along the two paths.

In [ ]:
####################################################
### Import Libraries and set up global variables ###
####################################################

from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Install packages only if needed
# Google Colab does not include every Python package
#
# Psi4 cannot be installed with pip. On Colab we first install conda
# (condacolab), which RESTARTS the session. After the restart, run
# this cell again and Psi4 will be installed with conda.
try:                  # if the package exists import it, otherwise install it
    import psi4
except ImportError:
    if IN_COLAB:
        !pip -q install condacolab
        import condacolab
        condacolab.install()
        !mamba install -q -y -c conda-forge psi4
    import psi4

try:
    import py3Dmol
except ImportError:
    !pip -q install py3Dmol
    import py3Dmol

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Plot style sheets from the book's GitHub repository
if IN_COLAB and not Path("styles/tufte.mplstyle").exists():
    !wget -q -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4/main/styles/tufte.mplstyle
    !wget -q -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4/main/styles/S2_classic2.mplstyle

if IN_COLAB:
    plt.style.use("styles/tufte.mplstyle")
else:
    plt.style.use("../../styles/tufte.mplstyle")

## Setting Up the Notebook

As in every chapter, the hidden **first code cell** imports *Psi4*, *py3Dmol*, *NumPy*, *pandas* and *Matplotlib* and applies the plot style sheet. This notebook needs no solvent library: the water is made of atoms.

```{admonition} Running this notebook in Google Colab
:class: dropdown
The first code cell checks whether the notebook is running in *Google Colab*. If it is, the cell **installs** *Psi4* for you. *Colab* does not come with *conda*, so the cell first installs it using a package called *condacolab*. This **restarts** the *Colab* session. When it has restarted, simply **run the first cell again**. It will then install *Psi4* with *conda*, install *py3Dmol* with *pip*, and download the plot style sheets from the book's *GitHub* repository. On your own computer, the cell just imports the packages.
```

```{warning}
This notebook takes a while: about 20 minutes on my computer. The cluster has 18 atoms, and every Hessian (there are several) needs about a hundred gradients. Start it, then go and make a cup of tea. In a teapot. With water molecules.
```

In [ ]:
psi4.set_memory("4 GB")
psi4.set_num_threads(4)

psi4.set_options({"scf_type": "pk"})

### Some Helpful Tools

We reuse the functions from the [first notebook](101-Solvent.ipynb) of this chapter, and add one more. `nearest_h()` finds the shortest distance from an atom to any hydrogen of a water molecule, which is how we will measure the hydrogen bonds to the chlorides. Run the cell below once.

In [ ]:
#| class: scroll-input
import logging

def new_output(filename):
    logger = logging.getLogger("psi4")
    for handler in logger.handlers[:]:
        if isinstance(handler, logging.FileHandler):
            handler.close()
            logger.removeHandler(handler)
    psi4.set_output_file(filename, False)

def show_molecule(mol):
    view = py3Dmol.view(width=400, height=300)
    view.addModel(mol.save_string_xyz_file(), "xyz")
    view.setStyle({"stick": {}, "sphere": {"scale": 0.25}})
    view.setBackgroundColor("#f2f2f2")
    view.zoomTo()
    view.show()

def bond_length(mol, i, j):
    xyz = mol.geometry().np * psi4.constants.bohr2angstroms
    return np.linalg.norm(xyz[i] - xyz[j])

def nearest_h(mol, i, first_water=6):
    return min(bond_length(mol, i, j) for j in range(first_water, mol.natom())
               if mol.symbol(j).upper() == "H")

def show_vibration(wfn, mode=0):
    vib = wfn.frequency_analysis
    vibrations = [i for i, kind in enumerate(vib["TRV"].data) if kind == "V"]
    vector = vib["x"].data[:, vibrations[mode]].reshape(-1, 3)
    mol = wfn.molecule()
    xyz = mol.geometry().np * psi4.constants.bohr2angstroms

    lines = [str(mol.natom()), "normal mode"]
    for i in range(mol.natom()):
        x, y, z = xyz[i]
        dx, dy, dz = vector[i]
        lines.append(f"{mol.symbol(i)} {x:.6f} {y:.6f} {z:.6f} {dx:.6f} {dy:.6f} {dz:.6f}")

    view = py3Dmol.view(width=400, height=300)
    view.addModel("\n".join(lines), "xyz", {"vibrate": {"frames": 10, "amplitude": 1.0}})
    view.setStyle({"stick": {}, "sphere": {"scale": 0.25}})
    view.setBackgroundColor("#f2f2f2")
    view.animate({"loop": "backAndForth"})
    view.zoomTo()
    view.show()

def c1_copy(mol):
    xyz = mol.geometry().np * psi4.constants.bohr2angstroms
    lines = [f"{int(mol.molecular_charge())} {mol.multiplicity()}"]
    for i in range(mol.natom()):
        x, y, z = xyz[i]
        lines.append(f"{mol.symbol(i)} {x:.10f} {y:.10f} {z:.10f}")
    lines.append("symmetry c1")
    lines.append("no_reorient")
    lines.append("no_com")
    return psi4.geometry("\n".join(lines))

def read_irc(filename):
    with open(filename) as f:
        lines = f.readlines()

    frames = []
    i = 0
    while i < len(lines) and lines[i].strip() != "":
        natom = int(lines[i])
        title = lines[i + 1].split()
        atoms = lines[i + 2:i + 2 + natom]
        symbols = [atom.split()[0] for atom in atoms]
        xyz = np.array([[float(v) for v in atom.split()[1:4]] for atom in atoms])
        xyz = xyz * psi4.constants.bohr2angstroms

        text = f"{natom}\n{lines[i + 1].strip()}\n"
        for symbol, (x, y, z) in zip(symbols, xyz):
            text = text + f"{symbol} {x:10.6f} {y:10.6f} {z:10.6f}\n"

        frames.append({"Step": int(float(title[3])),
                       "Energy": float(title[5]),
                       "text": text,
                       "xyz": xyz})
        i = i + 2 + natom
    return frames

def downhill(frames):
    energies = [frame["Energy"] for frame in frames]
    lowest = energies.index(min(energies))
    return frames[:lowest + 1]

def frame_to_molecule(frame, charge=-1):
    atom_lines = [line for line in frame["text"].split("\n")[2:] if line.strip() != ""]
    lines = [f"{charge} 1"] + atom_lines + ["symmetry c1", "no_reorient", "no_com"]
    return psi4.geometry("\n".join(lines))

def show_movie(frames):
    movie = "".join(frame["text"] for frame in frames)
    view = py3Dmol.view(width=400, height=300)
    view.addModelsAsFrames(movie, "xyz")
    view.setStyle({"stick": {}, "sphere": {"scale": 0.25}})
    view.setBackgroundColor("#f2f2f2")
    view.animate({"loop": "backAndForth", "interval": 100})
    view.zoomTo()
    view.show()
    return movie

| Function | What it does |
| :---- | :---- |
| `new_output(filename)` | Starts a new output file (and matching `.log` file) for the next job. |
| `show_molecule(mol)` | Draws a molecule in a 3-D viewer. |
| `bond_length(mol, i, j)` | Distance between atoms `i` and `j` in &Aring;ngstroms, counting from **0**. |
| `nearest_h(mol, i)` | The shortest distance from atom `i` to a water hydrogen. The water atoms start at atom 6 (counting from 0). |
| `show_vibration(wfn, mode)` | Animates one normal mode. Imaginary modes come first. |
| `c1_copy(mol)` | A copy of a molecule with no symmetry, which *Psi4* will not move or turn. |
| `read_irc(filename)` | Reads an IRC trajectory file into a list of frames (step, energy, coordinates in &Aring;ngstroms, *xyz* text). |
| `downhill(frames)` | Keeps the frames of one IRC direction up to the lowest energy. See the [first notebook](101-Solvent.ipynb) for why. |
| `frame_to_molecule(frame)` | Turns an IRC frame back into a *Psi4* molecule. |
| `show_movie(frames)` | Plays a list of frames as an animation and returns the multi-frame *xyz* text. |

## Building the Cluster

We start from the gas phase transition state of the [first notebook](101-Solvent.ipynb): carbon at the origin, the two chlorides on the *z*-axis at &plusmn;2.383 &Aring;, and the three hydrogens in the *xy* plane at 1.074 &Aring;. This time I put one hydrogen on the *y*-axis instead of the *x*-axis. You will see why in a moment.

Then the water. Each chloride gets **two** water molecules, 25&deg; on either side of the Cl&ndash;C&ndash;Cl axis. Each water points one O&ndash;H bond straight at its chloride, with a Cl&middot;&middot;&middot;H hydrogen bond of 2.2 &Aring; (a typical value). Its other hydrogen sits at the usual H&ndash;O&ndash;H angle of 104.5&deg;, **twisted** 60&deg; out of the plane. The two waters on a chloride are only about 2.7 &Aring; apart, close enough to hydrogen-bond to each other if they want to.

Why this arrangement? **Symmetry**, once again, but only a little of it. We build the waters around the bottom chloride, then make the waters around the top chloride as their **mirror images** through the *xy* plane. So the cluster keeps the mirror plane between the two chlorides, and, just as in gas phase, an optimization cannot break it: it will stay on the saddle point. Everything else is free. The twist breaks every other symmetry element, so the waters can turn and settle wherever they like (the box after the next cell explains why that matters). The cluster is *C<sub>s</sub>*.

In [ ]:
r_CCl = 2.383      # C-Cl in the gas phase transition state (Angstrom)
r_CH = 1.074       # C-H (Angstrom)
r_ClH = 2.20       # Cl...H-O hydrogen bond (Angstrom)
r_OH = 0.97        # O-H in water (Angstrom)
HOH = 104.5        # H-O-H angle in water (degrees)
angles = [-25, 25] # directions of the waters around each Cl (degrees from the axis)
twist = 60         # turn of each free H out of the xz plane (degrees)

def rotate_about(v, axis, degrees):
    a = np.radians(degrees)
    k = axis / np.linalg.norm(axis)
    return v * np.cos(a) + np.cross(k, v) * np.sin(a) + k * np.dot(k, v) * (1 - np.cos(a))

y_axis = np.array([0.0, 1.0, 0.0])
Cl1 = np.array([0.0, 0.0, -r_CCl])
atoms = [("Cl", Cl1), ("Cl", np.array([0.0, 0.0, r_CCl])), ("C", np.array([0.0, 0.0, 0.0]))]
for angle in [90, 210, 330]:
    theta = np.radians(angle)
    atoms.append(("H", np.array([r_CH * np.cos(theta), r_CH * np.sin(theta), 0.0])))

bottom = []
for angle in angles:
    a = np.radians(angle)
    d = np.array([np.sin(a), 0.0, -np.cos(a)])           # unit vector from Cl1 outwards
    H_bonded = Cl1 + r_ClH * d
    O = H_bonded + r_OH * d
    candidates = [O + r_OH * rotate_about(-d, y_axis, HOH), O + r_OH * rotate_about(-d, y_axis, -HOH)]
    H_free = max(candidates, key=np.linalg.norm)          # the one pointing outwards
    H_free = O + rotate_about(H_free - O, d, twist)       # twist it out of the plane
    bottom += [("O", O), ("H", H_bonded), ("H", H_free)]

top = [(s, xyz * np.array([1.0, 1.0, -1.0])) for s, xyz in bottom]   # mirror images
atoms = atoms + bottom + top

lines = ["-1 1"] + [f"{s} {x:.8f} {y:.8f} {z:.8f}" for s, (x, y, z) in atoms]
micro = psi4.geometry("\n".join(lines))
micro.update_geometry()

print("Number of atoms:          ", micro.natom())
print("Full point group:         ", micro.get_full_point_group())
print("Computational point group:", micro.schoenflies_symbol())
show_molecule(micro)

| Line | Notes |
| :---- | :---- |
| `angles = [-25, 25]` | The directions of the two waters on each chloride. Try `[0]` for one water per chloride, straight out along the axis. |
| `twist = 60` | How far each free hydrogen is turned out of the *xz* plane. |
| `def rotate_about(v, axis, degrees):` | Rotates the vector `v` about the direction `axis` by `degrees` (Rodrigues' rotation formula, if you want to look it up). We use it to place the free hydrogen of each water, and then to twist it. |
| `atoms = [("Cl", np.array([...])), ...]` | A list of (symbol, position) pairs. *NumPy* arrays let us add and scale positions like vectors. |
| `for angle in [90, 210, 330]:` | The three CH<sub>3</sub> hydrogens, the first one on the *y*-axis. |
| `bottom = []` | We build the waters on the **bottom** chloride only. |
| `d = np.array([np.sin(a), 0.0, -np.cos(a)])` | A **unit vector** (length 1) from the bottom chloride towards the water. |
| `H_bonded = chloride + r_ClH * d` | The hydrogen-bonded H, 2.2 &Aring; out along `d`. The oxygen is another 0.97 &Aring; further. |
| `candidates = [...]` | The free hydrogen could sit on either side of the O&ndash;H bond, at 104.5&deg; from it. |
| `max(candidates, key=np.linalg.norm)` | Picks the one that is **farthest** from the origin, so that it points away from the cluster instead of back at the chloride. |
| `rotate_about(H_free - O, d, twist)` | Turns the free hydrogen about the hydrogen bond, out of the *xz* plane. |
| `xyz * np.array([1.0, 1.0, -1.0])` | Changes the sign of *z*: the **mirror image** through the *xy* plane. These are the waters on the top chloride. |
| `f"{s} {x:.8f} {y:.8f} {z:.8f}"` | Eight decimals this time. In the [first notebook](101-Solvent.ipynb), six decimals were not enough for *Psi4*'s fussy point group label. |

*Psi4* finds 18 atoms and reports the point group as *C<sub>s</sub>*, both for the full and the computational group: only the mirror plane between the chlorides survived the twist, exactly as planned. In the picture you can see the [Cl&ndash;CH<sub>3</sub>&ndash;Cl]<sup>&minus;</sup> core in the middle and a pair of waters at each end, each pointing an O&ndash;H bond at its chloride.

```{admonition} Why only one mirror plane? (What went wrong the second time)
In my first version of this cluster, the waters sat 40&deg; from the axis, all flat in the *xz* plane, and the cluster was *C<sub>2v</sub>*. It optimized nicely, with symmetry holding it on the saddle point. But the frequencies told a different story: **nine** imaginary frequencies, the largest two at 412i cm<sup>&minus;1</sup>, besides the reaction mode at 207i. The symmetric structure was a saddle point in nine directions, most of them waters wanting to turn out of the plane.

The transition state search without symmetry then did what such a search does: it went **uphill** along the lowest mode, which was not the reaction but a water turning, and **downhill** along everything else, including the real reaction coordinate. After 100 steps it had slid all the way down to a chloride&ndash;methyl chloride complex (C&ndash;Cl 2.06 and 2.76 &Aring;), 105 kJ/mol **below** the symmetric structure. Most of that came from the waters: each pair had turned towards each other and formed a water&ndash;water hydrogen bond, a little ring of Cl<sup>&minus;</sup> and two waters.

- *The problem.* Too much symmetry. It held the transition state on the saddle point, as we wanted, but it also held the **waters** in a bad, flat arrangement.
- *The fix.* Keep only the one mirror plane that we need, between the two chlorides, and twist the waters so that no other symmetry survives. Then the optimizer can let the waters find their best arrangement while the core stays on the saddle point. I also moved the waters closer together (25&deg; from the axis instead of 40&deg;) so that each pair can hydrogen-bond to each other.
- *What I expect.* The optimization should end much lower in energy than the flat *C<sub>2v</sub>* structure, with the waters turned towards each other, and the frequencies should show only **one** imaginary frequency: the reaction. Then the transition state search has nothing left to do but confirm it.
```

The atoms are numbered as before: Cl1, Cl2 and C3 are *Python* atoms 0, 1 and 2, then the three CH<sub>3</sub> hydrogens, then the waters. So `bond_length(mol, 0, 2)` and `bond_length(mol, 1, 2)` are still our two C&ndash;Cl bonds.

## Symmetry Holds the Saddle Point

Now we optimize. As in the [first notebook](101-Solvent.ipynb), *Psi4* keeps the symmetry, including the mirror plane between the chlorides, so the optimizer finds the best structure **on** the saddle point. It is free to move the waters, as long as their mirror images move with them.

In [ ]:
psi4.core.clean()
psi4.core.clean_options()
psi4.set_options({"scf_type": "pk",
                  "opt_coordinates": "cartesian",
                  "geom_maxiter": 100})

new_output("micro_sym_opt.out")

E_micro_sym, hist_micro_sym = psi4.optimize("b3lyp/3-21g", molecule=micro, return_history=True)

print(f"Final energy: {E_micro_sym:.8f} Hartree")
print("Steps taken: ", len(hist_micro_sym["energy"]))
print("Point group: ", micro.schoenflies_symbol())
print()
print(f"C3-Cl1   {bond_length(micro, 0, 2):.4f}   C3-Cl2   {bond_length(micro, 1, 2):.4f} Angstrom")
print(f"Cl1...H  {nearest_h(micro, 0):.4f}   Cl2...H  {nearest_h(micro, 1):.4f} Angstrom")
show_molecule(micro)

| Line | Notes |
| :---- | :---- |
| `"opt_coordinates": "cartesian"` | Optimize in plain *x*, *y*, *z* coordinates. See the box below for why. |
| `"geom_maxiter": 100` | Cartesian optimizations can need more steps, so we allow up to 100. |
| `psi4.optimize("b3lyp/3-21g", molecule=micro, ...)` | The same level of theory as the [first notebook](101-Solvent.ipynb): B3LYP/3-21G. |
| `nearest_h(micro, 0)` | The Cl1&middot;&middot;&middot;H hydrogen bond length. |

The optimization took 26 steps and ended at &minus;1259.77902966 Hartree, still *C<sub>s</sub>*. The two C&ndash;Cl bonds are equal, as the mirror plane demands, at 2.3682 &Aring;: a little **shorter** than in the bare transition state (2.3829 &Aring;). The waters did **not** stay where we put them. Each pair turned towards each other until the oxygens were 2.64 &Aring; apart, close enough for a water&ndash;water hydrogen bond, and the Cl&middot;&middot;&middot;H hydrogen bonds relaxed from our 2.20 &Aring; to 2.389 &Aring;. Each chloride now sits in a little ring with its two waters.

```{admonition} Why Cartesian coordinates?
The first time I ran this notebook, the very first optimization stopped at once with an error from the optimizer: "Could not normalize vector". Here is what happened, from the optimizer's log file and its source code.

- *The problem.* The optimizer normally works in internal coordinates: bond lengths, angles and dihedrals. It decides which atoms are bonded from their distances. The hydrogen bonds (2.2 &Aring;) and the long C&ndash;Cl bonds (2.38 &Aring;) are too long to count as bonds, so the optimizer sees **five separate pieces**: the [Cl&ndash;CH<sub>3</sub>&ndash;Cl]<sup>&minus;</sup> core and four waters. To glue them together it keeps stretching its idea of a "bond" until everything is connected, and along the way it connected the two **chlorides** to each other (4.77 &Aring; apart).
- *Why that breaks.* Now there is an "angle" C&ndash;Cl1&ndash;Cl2. The carbon and the other chloride are in exactly the same direction from Cl1, so that angle is **zero**. To describe an angle, the optimizer needs the direction perpendicular to its two bonds, and two bonds that point the same way have no such direction. Division by zero, and the optimizer gives up before it takes a single step.
- *The fix.* `"opt_coordinates": "cartesian"`, as we used for the IRC in the [first notebook](101-Solvent.ipynb). With plain *x*, *y*, *z* coordinates there are no angles to go wrong. The price is that a Cartesian optimization may need more steps.
- *What I expect.* The optimization should now run. I have used Cartesian coordinates for every optimization in this notebook, the gas phase ones too, so that both systems are treated the same way.
```

### Is It a Transition State?

Let us check the vibrations. A transition state has exactly **one** imaginary frequency. If the waters still have somewhere better to go, there will be more, and our structure is a saddle point in more than one direction.

In [ ]:
psi4.core.clean()
new_output("micro_sym_freq.out")

E_tmp, wfn_micro_sym = psi4.frequency("b3lyp/3-21g", molecule=micro, return_wfn=True)

freqs = wfn_micro_sym.frequencies().to_array()
print("Imaginary frequencies:", freqs[freqs < 0])
print("Lowest five:          ", np.round(freqs[:5], 2))
show_vibration(wfn_micro_sym, 0)

| Line | Notes |
| :---- | :---- |
| `psi4.frequency(...)` | The Hessian by finite differences of exact gradients (symmetry saves some of the work), then the vibrational frequencies. |
| `show_vibration(wfn_micro_sym, 0)` | Animates the first (most imaginary) mode. |

There is one big imaginary frequency, 221.33i cm<sup>&minus;1</sup>. `show_vibration()` shows the familiar umbrella flip of the CH<sub>3</sub> group between the two chlorides, a little stiffer than in the bare transition state (211i). There are also three tiny ones, 20.8i, 4.2i and 1.7i cm<sup>&minus;1</sup>. Those are the floppy waters turning, and they are at the level of numerical noise (see the box after the next cell).

```{admonition} If there is more than one imaginary frequency
:class: dropdown
Use `show_vibration(wfn_micro_sym, 1)` (and `2`, &hellip;) to see the others. If they are waters twisting, the symmetric structure is not the true transition state. The search without symmetry in the next cell should take care of it, because it starts uphill only along the **lowest** mode, the reaction coordinate, and downhill along all the others. If it does not, try changing `angles` (for example `[-50, 50]`) and start again.
```

## A Real Transition State Search

Now the real thing, the one that failed in water with the continuum model. We take away the symmetry and let *Psi4*'s optimizer search for the transition state on its own. It calculates a Hessian at the start, walks **uphill** along the reaction mode and **downhill** along every other one. Then we check the frequencies again.

In [ ]:
ts_micro = c1_copy(micro)

psi4.core.clean()
psi4.core.clean_options()
psi4.set_options({"scf_type": "pk",
                  "opt_type": "ts",
                  "full_hess_every": 0,
                  "opt_coordinates": "cartesian",
                  "g_convergence": "gau",
                  "geom_maxiter": 100})

new_output("micro_ts.out")
E_ts_micro, hist_ts_micro = psi4.optimize("b3lyp/3-21g", molecule=ts_micro, return_history=True)

print(f"Final energy: {E_ts_micro:.8f} Hartree")
print("Steps taken: ", len(hist_ts_micro["energy"]))
print(f"Change from the symmetric structure: {(E_ts_micro - E_micro_sym) * psi4.constants.hartree2kJmol:.3f} kJ/mol")
print(f"C3-Cl1   {bond_length(ts_micro, 0, 2):.4f}   C3-Cl2   {bond_length(ts_micro, 1, 2):.4f} Angstrom")

psi4.core.clean()
psi4.core.clean_options()
psi4.set_options({"scf_type": "pk"})
new_output("micro_ts_freq.out")
E_tmp, wfn_ts_micro = psi4.frequency("b3lyp/3-21g", molecule=ts_micro, return_wfn=True)

freqs = wfn_ts_micro.frequencies().to_array()
print("Imaginary frequencies:", freqs[freqs < 0])
print("Larger than 50i:      ", freqs[freqs < -50])
print("Lowest five:          ", np.round(freqs[:5], 2))
show_vibration(wfn_ts_micro, 0)

| Line | Notes |
| :---- | :---- |
| `ts_micro = c1_copy(micro)` | The symmetric structure, with symmetry switched off. |
| `"opt_type": "ts"` | Search for a **transition state**. |
| `"full_hess_every": 0` | Calculate the Hessian once, at the start. |
| `"opt_coordinates": "cartesian"` | Cartesian coordinates again (see the box above). |
| `"g_convergence": "gau"` | The standard *Gaussian*-style convergence criteria. In the [butadiene](../08-Butadiene/081-Butadiene.ipynb) and [Diels-Alder](../09-DielsAlder/091-DielsAlder.ipynb) chapters we used the tighter `gau_tight`. See the box below for why not here. |
| `freqs[freqs < -50]` | Only the imaginary frequencies larger than 50i cm<sup>&minus;1</sup>. Smaller ones are usually the floppy waters and numerical noise (see the box below). |
| `psi4.frequency(...)` at the end | The check: exactly **one** imaginary frequency. Without symmetry this Hessian takes longer. |

The search stopped after 12 steps at &minus;1259.77903153 Hartree, only 0.005 kJ/mol below the symmetric structure. The C&ndash;Cl bonds are 2.3681 and 2.3683 &Aring;, still equal to 0.0002 &Aring;. Without any symmetry to help it, the optimizer found the same transition state. The check shows exactly **one** imaginary frequency larger than 50i, at 221.13i cm<sup>&minus;1</sup>. The other two, 9.9i and 0.9i, are the floppy waters again. This is our transition state with water, found the honest way.

```{admonition} Floppy waters (What went wrong the third time)
With only one mirror plane, the waters found a much better arrangement: each pair turned towards each other and formed a water&ndash;water hydrogen bond (O&middot;&middot;&middot;O 2.64 &Aring;). The energy came out 100 kJ/mol below the flat *C<sub>2v</sub>* cluster of the second run, and the C&ndash;Cl bonds stayed equal at 2.368 &Aring;. The frequencies showed **one** big imaginary frequency, 221i cm<sup>&minus;1</sup>, the umbrella flip of the reaction, and three tiny ones: 21i, 4i and 2i. Then the transition state search ran for 100 steps and **failed**.

- *The problem.* Look at the convergence table in `micro_ts.out`. The forces went down to 0.000001 Hartree/Bohr, far below the limit, and the energy stopped changing (it moved by 0.0003 kJ/mol in 100 steps). The C&ndash;Cl bonds never changed. The search had **found** the transition state, right where it started. But the steps never got small enough for the tight criteria (`gau_tight` wants every atom to move less than 0.00006 Bohr), because the optimizer kept nudging the waters back and forth.
- *Why it happened.* The waters are **floppy**. Turning a water a little about its hydrogen bond costs almost no energy, so those motions have tiny force constants (the vibrations at 4 and 2 cm<sup>&minus;1</sup>). With a tiny force constant, a tiny leftover force means a big step, and the optimizer chases its own tail. The three small imaginary frequencies are the same floppy motions. Frequencies this small are within the numerical noise of the DFT integration grid and of the finite-difference Hessian, and are usually ignored.
- *The fix.* Use the standard `gau` criteria, which allow steps of up to 0.0018 Bohr. Our structure already passes them.
- *What I expect.* The search should stop within a few steps, at the same structure, and the check should again show one imaginary frequency larger than 50i.
```

-----

## The IRC

Now we follow the path downhill in both directions, exactly as for the bare system in the [first notebook](101-Solvent.ipynb): Cartesian coordinates, steps of 0.3 amu<sup>&frac12;</sup> bohr, up to 20 points. Then we trim each direction at its lowest point. The function takes a `label` so that the gas phase runs below get their own file names.

In [ ]:
def run_irc(ts_mol, direction, label, points=20):
    psi4.core.clean()
    psi4.core.clean_options()
    psi4.set_options({"scf_type": "pk",
                      "opt_type": "irc",
                      "irc_direction": direction,
                      "irc_step_size": 0.3,
                      "irc_points": points,
                      "opt_coordinates": "cartesian"})

    mol = c1_copy(ts_mol)
    new_output(f"{label}_irc_{direction}.out")
    E = psi4.optimize("b3lyp/3-21g", molecule=mol)

    for f in Path(".").glob("irc_traj.*.xyz"):
        f.rename(f"{label}_irc_{direction}.xyz")
    return E

def irc_path(ts_mol, label, points=20):
    run_irc(ts_mol, "forward", label, points)
    run_irc(ts_mol, "backward", label, points)
    forward = read_irc(f"{label}_irc_forward.xyz")
    backward = read_irc(f"{label}_irc_backward.xyz")
    print(f"{label}: {len(forward)} forward and {len(backward)} backward frames in the files")
    forward = downhill(forward)
    backward = downhill(backward)
    print(f"{label}: {len(forward)} forward and {len(backward)} backward frames kept")
    return list(reversed(backward)) + forward[1:], len(backward)

micro_path, micro_n_back = irc_path(ts_micro, "micro")

| Line | Notes |
| :---- | :---- |
| `def run_irc(ts_mol, direction, label, points=20):` | The same function as in the [first notebook](101-Solvent.ipynb), with a `label` for the file names (`micro_irc_forward.xyz` and so on) and the most IRC points to take. `points=20` is a **default**: if we do not say, we get 20. |
| `def irc_path(ts_mol, label, points=20):` | Runs both directions, reads the trajectory files, trims each direction at its lowest energy and joins them into one path, from the far end of the backward run, through the transition state, to the far end of the forward run. |
| `return list(reversed(backward)) + forward[1:], len(backward)` | The path, and the number of backward frames. A *Python* function can return two things at once. |
| `micro_path, micro_n_back = irc_path(...)` | &hellip;and we catch them in two variables. |

Each direction wrote 9 frames, and `downhill()` kept 7 of them: six points (1.8 amu<sup>&frac12;</sup> bohr) on each side of the transition state. The last two frames in each file came after the lowest point and were trimmed, the same flat-bottom behaviour that we saw in the [first notebook](101-Solvent.ipynb). This time both directions stopped on their own.

### The Bottom of the Valley

As before, an ordinary optimization from each end of the trimmed path finds the complexes at the bottom of the valley: a chloride, hydrogen-bonded to its two waters, sitting at the back of a methyl chloride.

In [ ]:
def optimize_ends(path, label):
    ends = []
    for name, frame in [("backward", path[0]), ("forward", path[-1])]:
        psi4.core.clean()
        psi4.core.clean_options()
        psi4.set_options({"scf_type": "pk", "opt_coordinates": "cartesian", "geom_maxiter": 100})
        mol = frame_to_molecule(frame)
        new_output(f"{label}_complex_{name}.out")
        E = psi4.optimize("b3lyp/3-21g", molecule=mol)
        ends.append({"System": label, "Structure": f"complex ({name})", "Energy": E,
                     "C3-Cl1": bond_length(mol, 0, 2), "C3-Cl2": bond_length(mol, 1, 2)})
    return ends

micro_ends = optimize_ends(micro_path, "micro")
for end in micro_ends:
    print(f"{end['Structure']:20s}  C3-Cl1 {end['C3-Cl1']:.4f}  C3-Cl2 {end['C3-Cl2']:.4f} Angstrom   "
          f"{(end['Energy'] - E_ts_micro) * psi4.constants.hartree2kJmol:7.2f} kJ/mol from the TS")

| Line | Notes |
| :---- | :---- |
| `def optimize_ends(path, label):` | Optimizes the structures at both ends of a path and returns a list of results, one dictionary for each. |
| `"opt_coordinates": "cartesian"` | Cartesian coordinates again. |
| `"geom_maxiter": 100` | The valley is flat and the waters can rearrange, so we allow more steps than the usual 50. |

Both optimizations ended 4.95 kJ/mol below the transition state, with C&ndash;Cl bonds of 2.0560 and 2.7627 &Aring;: the two complexes are mirror images, as they should be. Each water pair kept its water&ndash;water hydrogen bond (O&middot;&middot;&middot;O 2.63&ndash;2.64 &Aring;), but the hydrogen bonds to the chlorides changed. On the chloride that has become the methyl chloride's chlorine (C&ndash;Cl 2.056 &Aring;) they lengthened to 2.46 &Aring;, and on the departing chloride (2.763 &Aring;) they shortened to 2.32&ndash;2.33 &Aring;. The waters hold on more tightly to the chloride that carries more of the negative charge.

-----

## The Same in Gas Phase

For a fair comparison, we now do exactly the same for the bare [Cl&ndash;CH<sub>3</sub>&ndash;Cl]<sup>&minus;</sup> in gas phase, with the same functions. This is a repeat of the [first notebook](101-Solvent.ipynb), packed into one cell: build the *D<sub>3h</sub>* structure, optimize it on the saddle point, follow the IRC and optimize the ends.

In [ ]:
lines = ["-1 1"] + [f"{s} {x:.8f} {y:.8f} {z:.8f}" for s, (x, y, z) in atoms[:6]]
bare = psi4.geometry("\n".join(lines))

psi4.core.clean()
psi4.core.clean_options()
psi4.set_options({"scf_type": "pk", "opt_coordinates": "cartesian", "geom_maxiter": 100})
new_output("bare_sym_opt.out")
E_ts_bare = psi4.optimize("b3lyp/3-21g", molecule=bare)
ts_bare = c1_copy(bare)
print(f"Bare TS: {E_ts_bare:.8f} Hartree   C3-Cl {bond_length(bare, 0, 2):.4f} Angstrom")

bare_path, bare_n_back = irc_path(ts_bare, "bare", points=5)
bare_ends = optimize_ends(bare_path, "bare")
for end in bare_ends:
    print(f"{end['Structure']:20s}  C3-Cl1 {end['C3-Cl1']:.4f}  C3-Cl2 {end['C3-Cl2']:.4f} Angstrom   "
          f"{(end['Energy'] - E_ts_bare) * psi4.constants.hartree2kJmol:7.2f} kJ/mol from the TS")

| Line | Notes |
| :---- | :---- |
| `atoms[:6]` | The first six atoms of our cluster: the two chlorides, the carbon and the three hydrogens. No water. |
| `irc_path(ts_bare, "bare", points=5)` | Only five IRC points for the bare system. See the box below for why. |
| `ts_bare = c1_copy(bare)` | We already know from the [first notebook](101-Solvent.ipynb) that the symmetric structure is the transition state (one imaginary frequency, 211i cm<sup>&minus;1</sup>), so we go straight to the IRC. |

The bare transition state came out at &minus;955.77602213 Hartree with C&ndash;Cl 2.3833 &Aring;, the same as in the first notebook to within the optimizer's tolerance. The IRC took five points in each direction and all of them were kept. The optimized complexes are 3.71 kJ/mol below the transition state, with C&ndash;Cl 2.0915 and 2.743 &Aring;. That agrees very well with the end of my *GAMESS* gas phase IRC (2.094 and 2.742 &Aring;, about &minus;3.7 kJ/mol).

```{admonition} Five points are enough (What went wrong the fourth time)
On my fourth run, everything with the waters worked, and then the **bare** IRC failed: "Could not converge geometry optimization in 300 iterations".

- *The problem.* The forward IRC took five good points in three or four small steps each, down to the bottom of the valley at 1.5 amu<sup>&frac12;</sup> bohr. Point six never converged. Its little constrained optimization bounced between two kinds of structure, 0.0004 Hartree apart, for 50 steps. Then the optimizer gave up on that point, started again, and bounced again, until all 300 steps allowed for the IRC were used up.
- *Why it happened.* It is the same flat valley bottom that we trimmed off in the [first notebook](101-Solvent.ipynb). Past the minimum, the next point on the sphere has nowhere downhill to go. In the first notebook the same calculation happened to wander to an end; this time it did not.
- *The fix.* We already know from the first notebook that the bare IRC reaches the bottom of the valley after five points (1.5 amu<sup>&frac12;</sup> bohr), and that the end optimizations then find the complexes. So we ask for only **five** points for the bare system. The cluster keeps its 20; with the waters it found the minimum and stopped on its own.
- *What I expect.* Five points in each direction, then the same complexes as in the first notebook.
```

-----

## Comparing the Two Paths

Now we put both paths into one table, in the same form as in the [second notebook](102-IRCData.ipynb): the stride along the path, the energy relative to the transition state, and the two C&ndash;Cl distances. The coordinates are already in the frames, so no *grep*-style cleaning is needed this time.

In [ ]:
def path_table(path, n_back, E_ts, label):
    rows = []
    for k, frame in enumerate(path):
        xyz = frame["xyz"]
        rows.append({"System": label,
                     "stride": round((k - (n_back - 1)) * 0.3, 2),
                     "Energy": frame["Energy"],
                     "E (kJ/mol)": (frame["Energy"] - E_ts) * psi4.constants.hartree2kJmol,
                     "C-Cl1": np.linalg.norm(xyz[0] - xyz[2]),
                     "C-Cl2": np.linalg.norm(xyz[1] - xyz[2])})
    return pd.DataFrame(rows)

paths = pd.concat([path_table(micro_path, micro_n_back, E_ts_micro, "microsolvated"),
                   path_table(bare_path, bare_n_back, E_ts_bare, "gas phase")])
paths = paths.reset_index(drop=True)
paths.to_json("sn2_micro_irc_B3LYP321G.json", orient="records", indent=2)

micro_table = paths[paths["System"] == "microsolvated"]
bare_table = paths[paths["System"] == "gas phase"]
display(micro_table.round(4))

| Line | Notes |
| :---- | :---- |
| `def path_table(path, n_back, E_ts, label):` | Turns a list of IRC frames into a table, one row per point. |
| `round((k - (n_back - 1)) * 0.3, 2)` | The **stride**: negative on the backward side, zero at the transition state, positive on the forward side. |
| `(frame["Energy"] - E_ts) * psi4.constants.hartree2kJmol` | The energy relative to **its own** transition state, in kJ/mol. |
| `pd.concat([...])` | Stacks the two tables. The `"System"` column tells them apart. |
| `paths[paths["System"] == "microsolvated"]` | Picks out the rows of one system. |
| `paths.to_json(...)` | Saves both paths as `sn2_micro_irc_B3LYP321G.json`. |

Let us plot the two energy profiles, and, if you ran the [first notebook](101-Solvent.ipynb) and its [data notebook](102-IRCData.ipynb), the energies in continuum water along the gas phase path as a third curve.

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6,5))

ax.plot(micro_table["stride"], micro_table["E (kJ/mol)"], color="black", linewidth=0.7, marker="o",
        markersize=4, markerfacecolor="white", markeredgecolor="black", markeredgewidth=0.5,
        label="4 explicit waters")
ax.plot(bare_table["stride"], bare_table["E (kJ/mol)"], color="black", linewidth=0.7, marker="o",
        markersize=4, markerfacecolor="#FFB3BA", markeredgecolor="black", markeredgewidth=0.5,
        label="gas phase")

if Path("sn2_irc_B3LYP321G.json").exists():
    ddx = pd.read_json("sn2_irc_B3LYP321G.json")
    ax.plot(ddx["stride"], ddx["E water (kJ/mol)"], color="black", linewidth=0.7, marker="o",
            markersize=4, markerfacecolor="#AEC6CF", markeredgecolor="black", markeredgewidth=0.5,
            label="continuum water (gas phase path)")

ax.set(title="Intrinsic Reaction Coordinate",
       ylabel=r"Energy $\left(\frac{kJ}{mol}\right)$",
       xlabel=r"Stride $\left(Bohr \cdot \sqrt{amu}\right)$")
ax.legend()
plt.show()

| Line | Notes |
| :---- | :---- |
| `if Path("sn2_irc_B3LYP321G.json").exists():` | Only draw the continuum curve if the file from the [second notebook](102-IRCData.ipynb) is in this folder. |
| `markerfacecolor="#AEC6CF"` | Pastel blue circles for the continuum model. |

Near the transition state, the two explicit models are almost the same: at a stride of 0.9 the cluster is 1.71 kJ/mol below its transition state and the bare system 1.53 kJ/mol. Further out the cluster pulls ahead, &minus;3.65 kJ/mol at 1.5 against &minus;3.09, and its complexes end up at &minus;4.95 kJ/mol against &minus;3.71. So four waters make the well a little deeper, by about a third. The continuum model is in another league: on the same gas phase structures it puts the points at a stride of 1.5 about 10.2 kJ/mol below the transition state. The continuum stabilizes the separating chloride far more than four waters do, because it surrounds it completely, and that is what lets the chloride drift away in my *GAMESS* water IRC. (The continuum curve jumps up beyond &plusmn;1.8. Those are the zig-zag points at the ends of the gas phase IRC, which we kept on purpose in the [first notebook](101-Solvent.ipynb). Only the smooth part between them is the real path.)

```{note}
The stride is the distance along the path in **mass-weighted** coordinates, and in the cluster the water molecules move too. So the same chemical change (say, the C&ndash;Cl bond growing by 0.3 &Aring;) can take a longer stride in the cluster than in the bare system. The bond lengths, plotted against each other, are a fairer way to compare the paths.
```

So let us plot one C&ndash;Cl bond against the other, the More O'Ferrall&ndash;Jencks plot of the [third notebook](103-IRCPlots.ipynb), for both systems.

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5,5))

low = min(paths["C-Cl1"].min(), paths["C-Cl2"].min()) - 0.05
high = max(paths["C-Cl1"].max(), paths["C-Cl2"].max()) + 0.05
ax.plot([low, high], [low, high], color="lightgray", linewidth=0.5, zorder=1)

ax.plot(micro_table["C-Cl2"], micro_table["C-Cl1"], color="black", linewidth=0.7, marker="o",
        markersize=4, markerfacecolor="white", markeredgecolor="black", markeredgewidth=0.5,
        label="4 explicit waters", zorder=2)
ax.plot(bare_table["C-Cl2"], bare_table["C-Cl1"], color="black", linewidth=0.7, marker="o",
        markersize=4, markerfacecolor="#FFB3BA", markeredgecolor="black", markeredgewidth=0.5,
        label="gas phase", zorder=2)

for ends, color in [(micro_ends, "white"), (bare_ends, "#FFB3BA")]:
    for end in ends:
        ax.plot(end["C3-Cl2"], end["C3-Cl1"], linestyle="none", marker="s", markersize=7,
                markerfacecolor=color, markeredgecolor="black", markeredgewidth=0.7, zorder=3)

ax.set(title="Intrinsic Reaction Coordinate",
       ylabel=r"1,3 C–Cl Bond Length $\left(\AA\right)$",
       xlabel=r"2,3 C–Cl Bond Length $\left(\AA\right)$",
       xlim=[low, high],
       ylim=[low, high])
ax.set_aspect("equal")
ax.legend()
plt.show()

| Line | Notes |
| :---- | :---- |
| `for ends, color in [...]:` | Adds the optimized complexes as **squares** in the colour of their system, so we can see how far the trimmed IRC is from the bottom of the valley. |
| `marker="s"` | A square marker. |

Both paths start on the diagonal at the transition state and leave it along almost the same curve. Near the transition state the waters hardly change the way the bonds are made and broken. The difference is at the ends. The squares show that the complexes with water are more separated than the bare ones: 2.056 and 2.763 &Aring; against 2.092 and 2.743 &Aring;. The new C&ndash;Cl bond is closer to the 1.901 &Aring; of free methyl chloride, and the chloride is a little further away. That is the same direction as the *GAMESS* continuum IRC, which ended at 1.96 and 3.11 &Aring;, but four waters get only a small part of the way there.

### Watching the Waters

Finally, the movie. Watch the water molecules as the chloride leaves. Do they follow it? Do they hand over their hydrogen bonds?

In [ ]:
micro_movie = show_movie(micro_path)
with open("micro_irc.xyz", "w") as f:
    f.write(micro_movie)

Watch the waters at each end. As one chloride moves in to form its bond, the hydrogen bonds to it get longer (from 2.39 to 2.46 &Aring; in the complex), and as the other chloride leaves, its waters hold on tighter (2.32&ndash;2.33 &Aring;). The pairs of waters stay paired all the way: they travel with their chlorides and do not swap partners.

## Barrier Heights

One more number that chemists care about: how high is the barrier from the complex at the bottom of the valley? Let us collect the transition states and complexes for both systems in one table and save it.

In [ ]:
rows = [{"System": "micro", "Structure": "TS", "Energy": E_ts_micro,
         "C3-Cl1": bond_length(ts_micro, 0, 2), "C3-Cl2": bond_length(ts_micro, 1, 2)},
        {"System": "bare", "Structure": "TS", "Energy": E_ts_bare,
         "C3-Cl1": bond_length(bare, 0, 2), "C3-Cl2": bond_length(bare, 1, 2)}] + micro_ends + bare_ends

summary = pd.DataFrame(rows)
E_ts = summary["System"].map({"micro": E_ts_micro, "bare": E_ts_bare})
summary["Barrier (kJ/mol)"] = (E_ts - summary["Energy"]) * psi4.constants.hartree2kJmol
summary.to_json("sn2_micro_summary_B3LYP321G.json", orient="records", indent=2)
summary.round(4)

| Line | Notes |
| :---- | :---- |
| `[...] + micro_ends + bare_ends` | The two transition states, then the four complexes from `optimize_ends()`. |
| `summary["System"].map({...})` | For each row, looks up the transition state energy of **its own** system. |
| `(E_ts - summary["Energy"]) * ...` | The barrier from each structure up to its transition state. Zero for the transition states themselves. |

From the complex, the barrier is 3.71 kJ/mol in gas phase and 4.95 kJ/mol with four waters. Water **raises** the barrier, by about a third. That is the right direction: chloride is stabilized by hydrogen bonds more than the transition state is, because in the transition state its charge is shared between two chlorines. *S<sub>N</sub>2* reactions of anions are much faster in gas phase than in water for exactly this reason. The numbers themselves are far too small. B3LYP is known to underestimate *S<sub>N</sub>2* barriers, and 3-21G is a very small basis set, so treat them as a trend, not as a prediction.

## Summary

Four hydrogen-bonded waters shortened the C&ndash;Cl bonds of the transition state a little (2.368 against 2.383 &Aring;), made the ion&ndash;dipole well about a third deeper (4.95 against 3.71 kJ/mol) and raised the barrier from that well by the same amount. The continuum model, on the same gas phase structures, deepened the well by much more (about 10 kJ/mol at a stride of 1.5). Water's main effect on this reaction comes from the bulk of the solvent, not from the first few molecules.

These are the operations that we have performed over the course of this activity.

1.	**Build** a *C<sub>s</sub>* cluster of the *S<sub>N</sub>2* transition state with four hydrogen-bonded waters, keeping only the mirror plane between the chlorides.
2.	**Optimize** it with that mirror plane holding it on the saddle point, and check its **vibrations**.
3.	**Search** for the transition state without symmetry, and confirm it has **one** imaginary frequency.
4.	**Follow the IRC** in both directions, trim the ends and **optimize** the complexes at the bottom of the valley.
5.	Do the same for the **bare** system in gas phase, and **compare** the energy profiles, the bond lengths and the barriers.

Explicit water gave us something the continuum model in *Psi4* could not: a real transition state search and a real IRC with water present. The continuum model gave us something that four waters cannot: the bulk of the solvent. Chemists often combine the two (a "cluster-continuum" model), but in *Psi4* that brings back the problem of the first notebook.

## A Challenge

### The Activity

How does the answer depend on the **number** of water molecules? Repeat the calculation with one water per chloride (`angles = [0]`), and with three (`angles = [-60, 0, 60]`, if you can make the waters fit). Plot the barrier from the complex against the number of waters.

```{admonition} Hints
:class: dropdown
- With `angles = [0]` there is one water per chloride. Keep the `twist`, so that the water can turn freely while the mirror plane between the chlorides holds the core on the saddle point.
- More waters means a bigger Hessian. The time grows quickly.
- Use `show_molecule()` after every optimization to check that the waters have not wandered off to hydrogen-bond to each other instead of to the chloride.
```

### The Report

Write a short report with a table of the barrier (from the complex) and the C&ndash;Cl distances in the complex for zero, one and two waters per chloride, and a plot of the energy profiles. Does the barrier converge towards the experimental value in water as more waters are added? (You will need to look that value up.)

```{warning} AI Statement
:class: dropdown
This page is new; it has no counterpart in my [previous web book on using *GAMESS*](https://blinkletter.github.io/UsingGAMESS). I asked an **LLM system** to write a page that models water with a few explicit water molecules, so that the transition state search and IRC that failed with the continuum model could be done in *Psi4*, and to compare the result with gas phase. The text, the *Python* code, the line-by-line explanations of it and the commentary were written by the LLM. The calculations were run by me on my own computer, and the numbers quoted in the text come from those calculations.
```